# AdDhakhiraCorpusAI - Assistant Bibliographique Web App

Configuration `default` : `gemma-4-12B-it` pour l'extraction, `Qwen3.6-35B-A3B` pour le raisonnement, retrieval dense FAISS avec `Qwen/Qwen3-Embedding-4B`.
La liste déroulante propose aussi `lite_version`, Gemini, ChatGPT/OpenAI ou Claude/Anthropic et remplacent uniquement les deux LLM ; le retrieval, les pages et la sortie restent identiques.

## Utilisation
1. Clique sur le bouton **Play** de la cellule `Initialiser`.
2. Attends le message `Initialisation terminee`.
3. Clique sur le bouton **Play** de la cellule `Ouvrir l'interface`.
4. Choisis le backend d'inférence, écris ta question puis clique sur **Envoyer**.


## Etape 1 - Initialiser
Clique sur **Play**. Cette étape installe AdDhakhira sur Google Drive, prépare les modèles locaux si demandé et construit l'index dense FAISS si nécessaire.
Tu peux changer les modèles via `MODEL_EXTRACTOR_ID`, `MODEL_REASONER_ID` et `EMBEDDING_MODEL_ID`.


In [ ]:
#@title Etape 1 - Initialiser
REPO_GIT_URL = "https://github.com/git-haddadz/AdDhakhiraCorpusAI.git" #@param {type:"string"}
REPO_DIR = "/content/AdDhakhiraCorpusAI" #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/AdDhakhiraCorpusAI" #@param {type:"string"}
MODEL_EXTRACTOR_ID = "gemma-4-12B-it" #@param {type:"string"}
MODEL_REASONER_ID = "Qwen3.6-35B-A3B" #@param {type:"string"}
LITE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ" #@param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}
OPENAI_MODEL = "gpt-4.1" #@param {type:"string"}
ANTHROPIC_MODEL = "claude-sonnet-4-20250514" #@param {type:"string"}
EMBEDDING_MODEL_ID = "Qwen/Qwen3-Embedding-4B" #@param {type:"string"}
DOWNLOAD_MODEL = False #@param {type:"boolean"}
BUILD_DENSE_INDEX = True #@param {type:"boolean"}
NUM_GPUS_EXTRACTOR = 1 #@param {type:"integer"}
NUM_GPUS_REASONER = 1 #@param {type:"integer"}

import os
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
repo_path = Path(REPO_DIR)
if not (repo_path / 'main.py').exists():
    if repo_path.exists():
        subprocess.run(['rm', '-rf', str(repo_path)], check=True)
    subprocess.run(['git', 'clone', REPO_GIT_URL, str(repo_path)], check=True)
os.chdir(repo_path)
deps_marker = repo_path / '.colab_deps_ok'
if not deps_marker.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', '-r', 'requirements.txt'], check=True)
    deps_marker.write_text('ok', encoding='utf-8')
    print('Dependances installees. Redemarrage automatique du runtime...')
    os.kill(os.getpid(), 9)

def local_model_dir(root: Path, model_id: str) -> Path:
    return root / model_id.strip('/').replace('/', '__')

def model_ref(model_dir: Path, model_id: str) -> str:
    return f'{model_dir}/' if (model_dir / 'config.json').exists() else model_id

project_dir = Path(DRIVE_DIR)
models_dir = project_dir / 'models'
extractor_model_dir = local_model_dir(models_dir, MODEL_EXTRACTOR_ID)
reasoner_model_dir = local_model_dir(models_dir, MODEL_REASONER_ID)
lite_model_dir = local_model_dir(models_dir, LITE_MODEL_ID)
embedding_model_dir = local_model_dir(models_dir, EMBEDDING_MODEL_ID)
index_dir = project_dir / 'vector_indexes'
output_dir = project_dir / 'outputs'
models_dir.mkdir(parents=True, exist_ok=True)
index_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_MODEL:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN', None)
    snapshot_download(repo_id=MODEL_EXTRACTOR_ID, local_dir=str(extractor_model_dir), local_dir_use_symlinks=False, token=token)
    if MODEL_REASONER_ID != MODEL_EXTRACTOR_ID:
        snapshot_download(repo_id=MODEL_REASONER_ID, local_dir=str(reasoner_model_dir), local_dir_use_symlinks=False, token=token)
    snapshot_download(repo_id=LITE_MODEL_ID, local_dir=str(lite_model_dir), local_dir_use_symlinks=False, token=token)
    snapshot_download(repo_id=EMBEDDING_MODEL_ID, local_dir=str(embedding_model_dir), local_dir_use_symlinks=False, token=token)

LOCAL_EXTRACTOR_MODEL = model_ref(extractor_model_dir, MODEL_EXTRACTOR_ID)
LOCAL_REASONER_MODEL = model_ref(reasoner_model_dir, MODEL_REASONER_ID)
LITE_MODEL = model_ref(lite_model_dir, LITE_MODEL_ID)
EMBEDDING_MODEL_REF = model_ref(embedding_model_dir, EMBEDDING_MODEL_ID)
BASE_CONFIG = {
    'JSON_INPUT_PATH': str(repo_path / 'database'),
    'TOP_K_CHUNKS': 8,
    'TOP_K_PAGES': 5,
    'MAX_MODEL_LEN_EXTRACTOR': 4096,
    'MIN_MODEL_LEN_REASONER': 8192,
    'NUM_GPUS_EXTRACTOR': int(NUM_GPUS_EXTRACTOR),
    'NUM_GPUS_REASONER': int(NUM_GPUS_REASONER),
    'VLLM_GPU_MEMORY_UTILIZATION': 0.90,
    'VLLM_MAX_NUM_BATCHED_TOKENS': None,
    'EMBEDDING_MODEL': EMBEDDING_MODEL_REF,
    'EMBEDDING_INDEX_DIR': str(index_dir),
    'VECTOR_INDEX_BACKEND': 'faiss',
    'DENSE_TOP_K': 40,
    'HYBRID_DENSE_WEIGHT': 0.40,
    'HYBRID_LEXICAL_WEIGHT': 0.60,
    'ENABLE_DENSE_RETRIEVAL': True,
    'ENABLE_HYBRID_RETRIEVAL': False,
    'TRANSLATE_TOP_CHUNKS_TO_FRENCH': False,
    'TRANSLATION_MAX_TOKENS': 4096,
    'AUTO_TRANSLATE_QUESTION_TO_ARABIC': False,
    'REASONER_TEMPERATURE': 0.2,
    'REASONER_TOP_P': 0.9,
    'REASONER_OUTPUT_MAX_TOKENS': 1200,
    'REASONER_CONTEXT_SAFETY_TOKENS': 8000,
    'JSON_GENERATION_MAX_RETRIES': 4,
    'JSON_GENERATION_MAX_TOKEN_MULTIPLIER': 4,
}

if BUILD_DENSE_INDEX:
    subprocess.run([
        sys.executable, '-m', 'src.vector_index',
        '--model', EMBEDDING_MODEL_REF,
        '--backend', 'faiss',
        '--json-input', str(repo_path / 'database'),
        '--output-dir', str(index_dir),
    ], check=True)

print('Initialisation terminee')
print('Repo:', repo_path)
print('Default extractor:', LOCAL_EXTRACTOR_MODEL)
print('Default reasoner:', LOCAL_REASONER_MODEL)
print('Lite model:', LITE_MODEL)
print('Embedding:', EMBEDDING_MODEL_REF)
print('Vector index:', index_dir)
print('Outputs:', output_dir)


## Etape 2 - Ouvrir l'interface
Clique sur **Play**. Une interface de chat apparaît juste en dessous.


In [ ]:
#@title Etape 2 - Ouvrir l'interface
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import gradio as gr

drive_dir = Path('/content/drive/MyDrive/AdDhakhiraCorpusAI')
session_dir = drive_dir / 'outputs' / f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
session_dir.mkdir(parents=True, exist_ok=True)
counter = {'n': 0}

def write_config_file(cfg):
    lines = [
        'from pathlib import Path',
        '',
        '# This file was generated by the notebook UI.',
        '# For manual local usage, copy src/config_template.py to src/config.py and edit it.',
        '',
    ]
    order = [
        'LLM_BACKEND', 'GEMINI_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY',
        'MODEL_EXTRACTOR_PATH', 'MODEL_REASONER_PATH', 'JSON_INPUT_PATH',
        'TOP_K_CHUNKS', 'TOP_K_PAGES', 'MAX_MODEL_LEN_EXTRACTOR', 'MIN_MODEL_LEN_REASONER',
        'NUM_GPUS_EXTRACTOR', 'NUM_GPUS_REASONER', 'VLLM_GPU_MEMORY_UTILIZATION',
        'VLLM_MAX_NUM_BATCHED_TOKENS', 'EMBEDDING_MODEL', 'EMBEDDING_INDEX_DIR',
        'VECTOR_INDEX_BACKEND', 'DENSE_TOP_K', 'HYBRID_DENSE_WEIGHT', 'HYBRID_LEXICAL_WEIGHT',
        'ENABLE_DENSE_RETRIEVAL', 'ENABLE_HYBRID_RETRIEVAL', 'TRANSLATE_TOP_CHUNKS_TO_FRENCH',
        'TRANSLATION_MAX_TOKENS', 'AUTO_TRANSLATE_QUESTION_TO_ARABIC', 'REASONER_TEMPERATURE',
        'REASONER_TOP_P', 'REASONER_OUTPUT_MAX_TOKENS', 'REASONER_CONTEXT_SAFETY_TOKENS',
        'JSON_GENERATION_MAX_RETRIES', 'JSON_GENERATION_MAX_TOKEN_MULTIPLIER',
    ]
    for key in order:
        value = cfg[key]
        if key in {'JSON_INPUT_PATH', 'EMBEDDING_INDEX_DIR'}:
            lines.append(f'{key} = Path({str(value)!r})')
        else:
            lines.append(f'{key} = {value!r}')
    config_path = Path('/content/AdDhakhiraCorpusAI/src/config.py')
    config_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

def clear_pipeline_modules():
    for name in list(sys.modules):
        if name == 'src.config' or name.startswith(('src.pipeline', 'src.llm_ops', 'src.llm_backend', 'src.retrieval')):
            sys.modules.pop(name, None)

def build_backend_config(backend, gemini_api_key, openai_api_key, anthropic_api_key):
    cfg = dict(BASE_CONFIG)
    if backend == 'gemini_api':
        if not (gemini_api_key or '').strip():
            raise ValueError('GEMINI_API_KEY est requis pour le backend Gemini.')
        cfg.update({
            'LLM_BACKEND': 'gemini_api',
            'MODEL_EXTRACTOR_PATH': GEMINI_MODEL,
            'MODEL_REASONER_PATH': GEMINI_MODEL,
            'GEMINI_API_KEY': gemini_api_key.strip(),
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': '',
        })
    elif backend == 'openai_api':
        if not (openai_api_key or '').strip():
            raise ValueError('OPENAI_API_KEY est requis pour le backend OpenAI.')
        cfg.update({
            'LLM_BACKEND': 'openai_api',
            'MODEL_EXTRACTOR_PATH': OPENAI_MODEL,
            'MODEL_REASONER_PATH': OPENAI_MODEL,
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': openai_api_key.strip(),
            'ANTHROPIC_API_KEY': '',
        })
    elif backend == 'anthropic_api':
        if not (anthropic_api_key or '').strip():
            raise ValueError('ANTHROPIC_API_KEY est requis pour le backend Anthropic.')
        cfg.update({
            'LLM_BACKEND': 'anthropic_api',
            'MODEL_EXTRACTOR_PATH': ANTHROPIC_MODEL,
            'MODEL_REASONER_PATH': ANTHROPIC_MODEL,
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': anthropic_api_key.strip(),
        })
    elif backend == 'lite_version':
        cfg.update({
            'LLM_BACKEND': 'default',
            'MODEL_EXTRACTOR_PATH': LITE_MODEL,
            'MODEL_REASONER_PATH': LITE_MODEL,
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': '',
        })
    else:
        cfg.update({
            'LLM_BACKEND': 'default',
            'MODEL_EXTRACTOR_PATH': LOCAL_EXTRACTOR_MODEL,
            'MODEL_REASONER_PATH': LOCAL_REASONER_MODEL,
            'GEMINI_API_KEY': '',
            'OPENAI_API_KEY': '',
            'ANTHROPIC_API_KEY': '',
        })
    return cfg

def ask(backend, gemini_api_key, openai_api_key, anthropic_api_key, question):
    question = (question or '').strip()
    if not question:
        return 'Saisis une question.', '', None
    try:
        cfg = build_backend_config(backend, gemini_api_key, openai_api_key, anthropic_api_key)
        write_config_file(cfg)
        clear_pipeline_modules()
        from src.pipeline import build_final_report
        from src.reporting import write_output_with_timing
        counter['n'] += 1
        out_path = session_dir / f'output_{counter["n"]}.html'
        t0 = time.time()
        report = build_final_report(question=question, translate_to_french=False, diagnostic_coherence=True)
        elapsed = time.time() - t0
        write_output_with_timing(report, elapsed_seconds=elapsed, output_path=str(out_path))
        status = f'Termine en {elapsed:.1f}s avec {backend}. HTML sauvegarde : {out_path}'
        return status, report, str(out_path)
    except Exception as exc:
        return f'Erreur: {exc}', '', None

def toggle_api_keys(backend):
    return (
        gr.update(visible=(backend == 'gemini_api')),
        gr.update(visible=(backend == 'openai_api')),
        gr.update(visible=(backend == 'anthropic_api')),
    )

with gr.Blocks(title='AdDhakhiraCorpusAI') as demo:
    gr.Markdown('## Assistant de recherche Malikite')
    backend = gr.Dropdown(
        choices=[('Default local vLLM', 'default'), ('Lite version', 'lite_version'), ('Gemini API', 'gemini_api'), ('ChatGPT / OpenAI API', 'openai_api'), ('Claude / Anthropic API', 'anthropic_api')],
        value='default',
        label='Backend inference',
    )
    gemini_api_key = gr.Textbox(label='GEMINI_API_KEY', type='password', visible=False)
    openai_api_key = gr.Textbox(label='OPENAI_API_KEY', type='password', visible=False)
    anthropic_api_key = gr.Textbox(label='ANTHROPIC_API_KEY', type='password', visible=False)
    question = gr.Textbox(label='Question', lines=4)
    submit = gr.Button('Envoyer', variant='primary')
    status = gr.Markdown()
    answer = gr.HTML()
    download = gr.File(label='Telecharger la reponse HTML')
    backend.change(toggle_api_keys, inputs=backend, outputs=[gemini_api_key, openai_api_key, anthropic_api_key])
    submit.click(ask, inputs=[backend, gemini_api_key, openai_api_key, anthropic_api_key, question], outputs=[status, answer, download])

print('Dossier de sortie:', session_dir)
demo.launch(share=True, debug=False)
